# Data Preparation

In [36]:
import pandas as pd

from src.compare import compare_transformations, compare_skewness, compare_distributions
from src.eda import understand_features
from src.enums import keys, prefixes, measurements
from src.transform import Transformation, transform

aq = pd.read_csv('04-01-open-meteo-2023-2026.csv')
rv = pd.read_csv('04-01-respiratory-virus-dashboard-2023-2026.csv')

## Stabilize air quality data

Further analysis showed that previously stabilized data become skewed again after aggregation. Also, previously extremely skewed columns can now be stabilized with transformations. We are applying stabilization to these columns again.

Features that cannot be stabilized would have been discretized.

### Compare skewness across different transformations

In [ ]:
compare_transformations(aq)

### Transform skewed features

In [ ]:
transformations = {
    keys['us_aqi_pm2_5 (USAQI)']: Transformation.SQRT,
    keys['us_aqi_pm10 (USAQI)']: Transformation.SQRT,
    keys['us_aqi_sulphur_dioxide (USAQI)']: Transformation.SQRT,
    keys['aerosol_optical_depth ()']: Transformation.SQRT,
    keys['nitrogen_dioxide (μg/m³)']: Transformation.SQRT,
    keys['pm10 (μg/m³)']: Transformation.SQRT,

    keys['us_aqi_carbon_monoxide (USAQI)']: Transformation.LOG1P,
    keys['pm2_5 (μg/m³)']: Transformation.LOG1P,
    keys['carbon_monoxide (μg/m³)']: Transformation.LOG1P,

    keys['us_aqi (USAQI)']: Transformation.YEO_JOHNSON,
    keys['us_aqi_ozone (USAQI)']: Transformation.YEO_JOHNSON,
    keys['dust (μg/m³)']: Transformation.YEO_JOHNSON,
    keys['sulphur_dioxide (μg/m³)']: Transformation.YEO_JOHNSON,
}
aq_transformed = transform(
    aq,
    transformations,
    prefixes['04_01_data_preparation_']
)

### Skewness comparison

In [ ]:
compare_skewness(aq, aq_transformed, transformations)

### Distribution comparison

In [ ]:
compare_distributions(
    aq,
    aq_transformed,
    [
        keys['us_aqi_pm2_5 (USAQI)'], # square root transformation.
        keys['us_aqi_sulphur_dioxide (USAQI)'], # square root transformation.
        keys['pm2_5 (μg/m³)'], # log transformation.
        keys['dust (μg/m³)'] # Yeo-Johnson transformation.
    ]
)

## Discretize categorical features

Discretize categorical columns so that they can be correlated with numeric columns.

### Determine value counts for each column

In [ ]:
value_counts = pd.concat([
    rv[keys['COV_TP_LEVEL']].value_counts(),
    rv[keys['FLU_TP_LEVEL']].value_counts(),
    rv[keys['RSV_TP_LEVEL']].value_counts(),
], axis=1)
value_counts.columns = [keys['COV_TP_LEVEL'], keys['FLU_TP_LEVEL'], keys['RSV_TP_LEVEL']]

value_counts

### Convert discrete strings to ordinal values

In [ ]:
rv_discretized = rv.copy();

level_order = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']

rv_discretized[keys['COV_TP_LEVEL']] = pd.Categorical(
    rv_discretized[keys['COV_TP_LEVEL']],
    categories=level_order,
    ordered=True
)
rv_discretized[keys['FLU_TP_LEVEL']] = pd.Categorical(
    rv_discretized[keys['FLU_TP_LEVEL']],
    categories=level_order,
    ordered=True
)
rv_discretized[keys['RSV_TP_LEVEL']] = pd.Categorical(
    rv_discretized[keys['RSV_TP_LEVEL']],
    categories=level_order,
    ordered=True
)

rv_preview = rv_discretized.copy()
rv_preview[keys['COV_TP_LEVEL'] + '_order'] = rv_preview[keys['COV_TP_LEVEL']].cat.codes
rv_preview[keys['FLU_TP_LEVEL'] + '_order'] = rv_preview[keys['FLU_TP_LEVEL']].cat.codes
rv_preview[keys['RSV_TP_LEVEL'] + '_order'] = rv_preview[keys['RSV_TP_LEVEL']].cat.codes

rv_preview.head(3)

## Stitch data
Combine datasets through their SEASON, WEEKENDING, MMWR_WEEK, MMWR_YEAR features.

In [ ]:
df = pd.merge(
    aq_transformed,
    rv_discretized,
    on=[
        keys['SEASON'],
        keys['WEEKENDING'],
        keys['MMWR_WEEK'],
        keys['MMWR_YEAR']
    ],
    how='inner'
)
df_preview = df.copy()
print(f'Row length: {len(df_preview)}')
df_preview.head(3)

## Understand features

In [39]:
measurements = {
    keys['us_aqi (USAQI)']: measurements['ordinal'],
    keys['us_aqi_pm2_5 (USAQI)']: measurements['ordinal'],
    keys['us_aqi_pm10 (USAQI)']: measurements['ordinal'],
    keys['us_aqi_nitrogen_dioxide (USAQI)']: measurements['ordinal'],
    keys['us_aqi_carbon_monoxide (USAQI)']: measurements['ordinal'],
    keys['us_aqi_ozone (USAQI)']: measurements['ordinal'],
    keys['us_aqi_sulphur_dioxide (USAQI)']: measurements['ordinal'],
    keys['uv_index_clear_sky ()']: measurements['ratio'],
    keys['uv_index ()']: measurements['ratio'],
    keys['dust (μg/m³)']: measurements['ratio'],
    keys['aerosol_optical_depth ()']: measurements['ratio'],
    keys['ozone (μg/m³)']: measurements['ratio'],
    keys['sulphur_dioxide (μg/m³)']: measurements['ratio'],
    keys['nitrogen_dioxide (μg/m³)']: measurements['ratio'],
    keys['pm2_5 (μg/m³)']: measurements['ratio'],
    keys['carbon_monoxide (μg/m³)']: measurements['ratio'],
    keys['pm10 (μg/m³)']: measurements['ratio'],
    keys['SEASON']: measurements['nominal'],
    keys['WEEKENDING']: measurements['interval'],
    keys['MMWR_WEEK']: measurements['interval'],
    keys['MMWR_YEAR']: measurements['interval'],
    keys['COV_POSITIVES']: measurements['ratio'],
    keys['COV_TOTAL_TESTS']: measurements['ratio'],
    keys['COV_TP']: measurements['ratio'],
    keys['COV_TP_LEVEL']: measurements['ordinal'],
    keys['FLU_POSITIVES']: measurements['ratio'],
    keys['FLU_TOTAL_TESTS']: measurements['ratio'],
    keys['FLU_TP']: measurements['ratio'],
    keys['FLU_TP_LEVEL']: measurements['ordinal'],
    keys['RSV_POSITIVES']: measurements['ratio'],
    keys['RSV_TOTAL_TESTS']: measurements['ratio'],
    keys['RSV_TP']: measurements['ratio'],
    keys['RSV_TP_LEVEL']: measurements['ordinal'],
    keys['TOTAL_DEATHS']: measurements['ratio'],
    keys['COV_DEATHS']: measurements['ratio'],
    keys['FLU_DEATHS']: measurements['ratio'],
    keys['RSV_DEATHS']: measurements['ratio'],
    keys['COV_DEATHS_PER']: measurements['ratio'],
    keys['FLU_DEATHS_PER']: measurements['ratio'],
    keys['RSV_DEATHS_PER']: measurements['ratio'],
}
descriptions = {
    keys['us_aqi (USAQI)']: "Air Quality Index is the overall standardized levels of health risk or air pollution.",
    keys['us_aqi_pm2_5 (USAQI)']: "AQI for PM2.5 particles.",
    keys['us_aqi_pm10 (USAQI)']: "AQI for PM10 particles.",
    keys['us_aqi_nitrogen_dioxide (USAQI)']: "AQI for Nitrogen Dioxide.",
    keys['us_aqi_carbon_monoxide (USAQI)']: "AQI for Carbon Monoxide.",
    keys['us_aqi_ozone (USAQI)']: "AQI for Ozone.",
    keys['us_aqi_sulphur_dioxide (USAQI)']: "AQI for Sulphur Dioxide.",
    keys['uv_index_clear_sky ()']: "Measurement of the intensity of ultraviolet radiation from the sun under clear sky conditions, high levels can cause sunburn and skin cancer.",
    keys['uv_index ()']: "Measurement of the intensity of ultraviolet radiation from the sun, high levels can cause sunburn and skin cancer.",
    keys['dust (μg/m³)']: "Dust particles in the air that can cause coughing, wheezing, and asthma attacks.",
    keys['aerosol_optical_depth ()']: "Measure of how much sunlight is blocked by airborne particles. High AOD indicates high concentration of particulate matters (PM2.5 and PM10).",
    keys['ozone (μg/m³)']: "Created by chemical reactions between oxides of nitrogen and sunlight, it is a powerful lung irritant that can trigger asthma.",
    keys['sulphur_dioxide (μg/m³)']: "Produced by burning fossil fuels, it constricts airways and is particularly dangerous for people with asthma.",
    keys['nitrogen_dioxide (μg/m³)']: "Vehicle emissions can cause airway inflammation.",
    keys['pm2_5 (μg/m³)']: "Tiny particles that can lodge deep in the lungs and enter the bloodstream.",
    keys['carbon_monoxide (μg/m³)']: "Odorless gas from combustion, reduces oxygen delivery to organs.",
    keys['pm10 (μg/m³)']: "Includes dust, pollen, mold and affects the upper respiratory tract.",
    keys['SEASON']: 'The annual surveillance cycle starting in summer (MMWR week 26).',
    keys['WEEKENDING']: 'The Saturday date marking the end of the CDC reporting week.',
    keys['MMWR_WEEK']: 'Standardized CDC week number (1 to 52 or 53).',
    keys['MMWR_YEAR']: 'The calendar year associated with the reporting week.',
    keys['COV_POSITIVES']: 'Count of positive COVID-19 lab tests for the week.',
    keys['COV_TOTAL_TESTS']: 'Total volume of COVID-19 lab tests performed.',
    keys['COV_TP']: 'The percentage of COVID-19 tests that were positive. The formula is (COV_POSITIVES / COV_TOTAL_TESTS) * 100',
    keys['COV_TP_LEVEL']: 'Positivity level based on 5 seasons of COVID-19 data.',
    keys['FLU_POSITIVES']: 'Count of positive Influenza lab tests for the week.',
    keys['FLU_TOTAL_TESTS']: 'Total volume of Influenza lab tests performed.',
    keys['FLU_TP']: 'The percentage of Influenza tests that were positive. The formula is (FLU_POSITIVES / FLU_TOTAL_TESTS) * 100',
    keys['FLU_TP_LEVEL']: 'Positivity level based on 5 seasons of Influenza data.',
    keys['RSV_POSITIVES']: 'Count of positive RSV lab tests for the week.',
    keys['RSV_TOTAL_TESTS']: 'Total volume of RSV lab tests performed.',
    keys['RSV_TP']: 'The percentage of RSV tests that were positive. The formula is (RSV_POSITIVES / RSV_TOTAL_TESTS) * 100',
    keys['RSV_TP_LEVEL']: 'Positivity level based on 5 seasons of RSV data.',
    keys['TOTAL_DEATHS']: 'Number of all-cause deaths each week.',
    keys['COV_DEATHS']: 'Number of COVID-19 associated deaths each week.',
    keys['FLU_DEATHS']: 'Number of influenza-associated deaths each week.',
    keys['RSV_DEATHS']: 'Number of RSV-associated deaths each week.',
    keys['COV_DEATHS_PER']: 'Percentage of all-cause deaths attributed to COVID-19.',
    keys['FLU_DEATHS_PER']: 'Percentage of all-cause deaths attributed to Influenza.',
    keys['RSV_DEATHS_PER']: 'Percentage of all-cause deaths attributed to RSV.',
}

understand_features(df, measurements, descriptions)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 40 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   SEASON                           138 non-null    object  
 1   WEEKENDING                       138 non-null    object  
 2   MMWR_WEEK                        138 non-null    int64   
 3   MMWR_YEAR                        138 non-null    int64   
 4   us_aqi (USAQI)                   138 non-null    float64 
 5   us_aqi_pm2_5 (USAQI)             138 non-null    float64 
 6   us_aqi_pm10 (USAQI)              138 non-null    float64 
 7   us_aqi_nitrogen_dioxide (USAQI)  138 non-null    int64   
 8   us_aqi_carbon_monoxide (USAQI)   138 non-null    float64 
 9   us_aqi_ozone (USAQI)             138 non-null    float64 
 10  us_aqi_sulphur_dioxide (USAQI)   138 non-null    float64 
 11  uv_index ()                      138 non-null    float64 
 12  uv_index

,Measurement Type,Description
us_aqi (USAQI),ordinal,Air Quality Index is the overall standardized ...
us_aqi_pm2_5 (USAQI),ordinal,AQI for PM2.5 particles.
us_aqi_pm10 (USAQI),ordinal,AQI for PM10 particles.
us_aqi_nitrogen_dioxide (USAQI),ordinal,AQI for Nitrogen Dioxide.
us_aqi_carbon_monoxide (USAQI),ordinal,AQI for Carbon Monoxide.
us_aqi_ozone (USAQI),ordinal,AQI for Ozone.
us_aqi_sulphur_dioxide (USAQI),ordinal,AQI for Sulphur Dioxide.
uv_index_clear_sky (),ratio,Measurement of the intensity of ultraviolet ra...
uv_index (),ratio,Measurement of the intensity of ultraviolet ra...
dust (μg/m³),ratio,Dust particles in the air that can cause cough...


## Assess features